In [1]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Real_State"
datos_finales = []   # Se define fuera del try para que siempre exista
driver = None        # Se define fuera del try para poder cerrarlo con seguridad

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"  # Ruta del binario de Chrome

# Argumentos de estabilidad para Docker (SIN --headless para ver el navegador en VNC)
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print("🌐 Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y PAUSA HUMANA ---
    # Búsqueda de ropa en Amazon
    driver.get("https://www.amazon.es/s?k=ropa+de+marca")
    
    # 🛑 INTERVENCIÓN HUMANA (Requisito Semana 6)
    print("\n⚠️ ACCIÓN REQUERIDA:")
    print("1. Ve a la pestaña del escritorio del contenedor (VNC).")
    print("2. Verifica si aparece un Captcha, acepta las cookies o simplemente haz scroll.")
    print("3. Una vez que veas el listado de ropa correctamente, regresa aquí.")
    input("\n👉 Presiona ENTER en esta celda cuando estés listo para continuar...")
    print("Continuando con la extracción...\n")

    # --- PASO 3: EXTRACCIÓN DE DATOS ---
    limite_paginas = 3

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        bloques = driver.find_elements(
            By.CSS_SELECTOR, "div[data-component-type='s-search-result']"
        )

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.TAG_NAME, "h2").text
                precio = bloque.find_element(By.CSS_SELECTOR, ".a-price-whole").text

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "categoria": "Ropa",
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
            except:
                # Si el bloque no tiene nombre o precio, se salta
                continue

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.CLASS_NAME, "s-pagination-next")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(4) # Espera a que cargue el JS de la siguiente página
        except:
            print("No se encontró el botón siguiente o ya es la última página.")
            break

    print(f"\n✅ Extracción terminada: {len(datos_finales)} productos encontrados.")
    
    # 🔍 VALIDACIÓN: Imprimir al menos 3 productos
    print("\n--- Muestra de Validación (Primeros 3 productos) ---")
    for i, prod in enumerate(datos_finales[:3]):
        print(f"{i+1}. {prod['identificador']} - €{prod['valor']}")
    print("----------------------------------------------------\n")

except Exception as e:
    print(f"❌ Error en Selenium: {e}")

finally:
    # Cierra el navegador
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 4: GUARDAR EN MONGODB ---
try:
    # Mantenemos las variables solicitadas
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["G_RealEstate"]

    if datos_finales:
        for d in datos_finales:
            # Limpia el valor antes de convertirlo
            v_limpio = str(d["valor"]).replace(".", "").replace(",", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print("💾 Datos de ropa cargados en MongoDB correctamente.")
    else:
        print("⚠️ No hay datos para guardar.")

except Exception as e:
    print(f"❌ Error en MongoDB: {e}")

🧹 Limpieza de procesos y temporales completada.
🌐 Navegador iniciado correctamente.

⚠️ ACCIÓN REQUERIDA:
1. Ve a la pestaña del escritorio del contenedor (VNC).
2. Verifica si aparece un Captcha, acepta las cookies o simplemente haz scroll.
3. Una vez que veas el listado de ropa correctamente, regresa aquí.



👉 Presiona ENTER en esta celda cuando estés listo para continuar... 


Continuando con la extracción...

--- Procesando Página 1 ---
--- Procesando Página 2 ---
--- Procesando Página 3 ---

✅ Extracción terminada: 166 productos encontrados.

--- Muestra de Validación (Primeros 3 productos) ---
1. Tommy Hilfiger Adaptive - €44
2. Tommy Hilfiger Adaptive - €40
3. iClosam - €14
----------------------------------------------------

💾 Datos de ropa cargados en MongoDB correctamente.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, avg

# 1. Conexion con el Schema (usando la colección solicitada)
spark = SparkSession.builder \
    .appName("Analisis_Amazon_Ropa_S6") \
    .config("spark.mongodb.read.connection.uri", "mongodb://mongodb:27017/TiendaBigData.G_RealEstate") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

# 2. Transformación de Negocio: Extraer la MARCA
# En Amazon, la primera palabra del título suele ser la marca (Levis, Puma, Adidas, etc.)
df_transformado = df.withColumn("marca", split(col("identificador"), " ")[0])

# 3. Limpieza: Filtrar ropa con precios en cero o erróneos (solo tomaremos > 5 euros)
# Además nos aseguramos de filtrar solo la categoría 'Ropa' por si hay basura anterior en la BD
df_validado = df_transformado.filter((col("valor") > 5) & (col("categoria") == "Ropa"))

# --- EVIDENCIA SOLICITADA: Total de productos procesados ---
total_procesados = df_validado.count()
print(f"\n📊 TOTAL DE PRODUCTOS PROCESADOS PARA EL REPORTE: {total_procesados}\n")

# 4. Agregación: Precio promedio por Marca
reporte_marcas = df_validado.groupBy("marca").agg(
    avg("valor").alias("precio_promedio")
).orderBy(col("precio_promedio").desc())

print("--- PRECIO PROMEDIO POR MARCA DE ROPA ---")
reporte_marcas.show(15, truncate=False)


📊 TOTAL DE PRODUCTOS PROCESADOS PARA EL REPORTE: 166

--- PRECIO PROMEDIO POR MARCA DE ROPA ---
+---------+------------------+
|marca    |precio_promedio   |
+---------+------------------+
|SUTRAN   |139.0             |
|Armani   |100.0             |
|LEONE    |62.0              |
|Marc     |44.31481481481482 |
|Gorsenia |44.0              |
|Tommy    |39.333333333333336|
|ZITY     |39.0              |
|Neer     |39.0              |
|DANISH   |39.0              |
|SALEWA   |34.0              |
|PUMA     |32.75             |
|SwissWell|32.0              |
|KARL     |32.0              |
|Dubinik  |31.0              |
|PJ       |31.0              |
+---------+------------------+
only showing top 15 rows

